In [4]:
import ipywidgets as widgets
from time import sleep
import random
from IPython.display import display

from enum import Enum

#---- Colors ----
colors = ["red", "blue", "green", "yellow"] #Enum

normalColors = {"red" : "#820707",
                "blue" : "#071782",
                "green" : "#055c0d",
                "yellow" : "#5c5a05"}

brightColors = {"red" : "#D51212",
                "blue" : "#132BC1",
                "green" : "#11bb22",
                "yellow" : "#bebb0b"}

#---- Game configuration ----
gameSequence = [] #List of colors the game is choosing
userSequence = []
score = 0

class gameStatus(Enum):
    waiting = "waiting"
    playing = "playing"
    showing = "showing"
    
status = gameStatus.waiting




#---- UI ----

#Instruction label
label = widgets.HTML(
    value = "<h1>Presiona <b>JUGAR</b> para comenzar</h1>",
    layout = widgets.Layout(margin = "10px")
)


#Button
play_button = widgets.Button(
    description = "PLAY",
    button_style = "success",
    layout = widgets.Layout(width="150px", height="40px", margin="10px")
)

#Game buttons
game_buttons = {}

for color in colors:
    game_buttons[color] = widgets.Button(
        description = "",
        layout = widgets.Layout(width="100px", height="100px", margin="5px")
    )
    #Assign color for each button
    game_buttons[color].style.button_color = normalColors[color]

#Button rows
row1 = widgets.HBox([game_buttons["red"], game_buttons["blue"]])
row2 = widgets.HBox([game_buttons["green"], game_buttons["yellow"]])

#Container for UI elements
gameContainer = widgets.VBox(
    [label, row1, row2,play_button],
    layout = widgets.Layout(align_items = "center", background_color = "#222222", padding = "15px", border_radius = "10px", width = "300px"))


#Game Logic and functions

def shineColor (color):
    game_buttons[color].style.button_color = brightColors[color]

def restoreColor (color):
    game_buttons[color].style.button_color = normalColors[color]

def changeColor (color, mode):
    if mode == "shine":
        game_buttons[color].style.button_color = brightColors[color]
        print("Shining")
    elif mode == "restore":
        game_buttons[color].style.button_color = normalColors[color]
        print("Restoring")
    else:
        print("Not valid mode")
        game_buttons[color].style.button_color = normalColors[color]

def destellar (color):
    changeColor(color, "shine")
    sleep(0.4)
    changeColor(color, "restore")
    sleep(0.2)

def showSequence ():
    global status, userSequence
    status = gameStatus.showing
    userSequence = [] #Reset user sequence
    label.value = "<h1>Atento a la secuencia mostrada</h1>"

    sleep(0.8)

    for color in gameSequence:
        destellar(color)

    status = gameStatus.playing
    label.value = f"<h1>Tu turno Nivel:{len(gameSequence)} </h1>"

def addSequence ():
    newColor = random.choice(colors)
    gameSequence.append(newColor)
    showSequence()

def startGame (buttonStatus):
    global gameSequence, score
    gameSequence = [] #Empty previous sequence
    score = 0
    play_button.layout.display = "none" #Hide button after starting
    addSequence()

def whenButtonPressed (buttonPresed):
    global status, score, userSequence

    if status != gameStatus.playing:
        return

    colorPressed = None

    for name,widget in game_buttons.items():
        if widget == buttonPresed:
            colorPressed = name
            break

    changeColor(colorPressed, "shine")
    sleep(0.2)
    changeColor(colorPressed, "restore")

    userSequence.append(colorPressed) #This is callled every time the button is pressed to compare the same index

    stage = len(userSequence) -1 #At what part of the sequence is the player at

    if userSequence[stage] == gameSequence[stage]:
        if len(userSequence) == len(gameSequence):
            score += 1
            label.value = "<h1>Muy bien. Siguiente nivel </h3>"
            status = gameStatus.waiting
            sleep(1)
            addSequence()
    else:
        status = gameStatus.waiting
        label.value = f"<h1>Perdiste. Puntuación final: {score}</h1>"
        play_button.description = "REINCIAR"
        play_button.layout.display = "block"

play_button.on_click(startGame)

for color in colors:
    game_buttons[color].on_click(whenButtonPressed)

display(gameContainer)
